# Aufgabe 4.1 und 4.2: Bohrpfahl

In [ ]:
using DotMaps
using CairoMakie
using LinearAlgebra

using MMJMesh.Plots
using MMJMesh.Mathematics

## Problemparameter

In [ ]:
# Mehrfachvariable
p = DotMap()

# Basiswerte
p.E = 35e9
p.d = 0.8
p.l = 20
p.F = 2e6
p.ρ = 2500
p.g = 9.81
p.S = 120e6
p.c = 1.75e7

# Abgeleitete Werte
p.Ω = 0 .. p.l
p.C = p.c * π * p.d
p.A = π * p.d^2 / 4
p.EA = p.E * p.A
p.n = p.ρ * p.g * p.A;

## Exakte Lösung

In [ ]:
a1 = 3 + 7π
a2 = 7π - 3
x = parameter(p.Ω)

u = exp(-x / 20) / (3500000 * π * (a1 * ℯ^2 - a2)) * (
        25000 * (a1 * ℯ^2 + a2 * exp(x / 10))
        -
        2943 * π * ℯ * (exp(x / 10) + 1)
        +
        981 * π * exp(x / 20) * (a1 * ℯ^2 - a2)
    )

N = p.EA * u'

fig = Figure()
fplot!(Axis(fig[1, 1], title="Verschiebung u"), u, npoints=100)
fplot!(Axis(fig[2, 1], title="Normalkraft N"), N, npoints=100)
fig

## Schwache Form

In [ ]:
a(u, δu) = integrate(p.EA * u' * δu' + p.C * u * δu, p.Ω) + p.S * u(p.l) * δu(p.l)
b(δu) = integrate(p.n * δu, p.Ω) + p.F * δu(0)

## Numerische Lösung

In [ ]:
function pile_fem(p, k)
    phis = hatfunctions(range(p.Ω, k + 1))
    K = [a(phiᵢ, phiⱼ) for phiᵢ = phis, phiⱼ = phis]
    r = [b(phiᵢ) for phiᵢ = phis]
    uHat = K \ r
    uh = sum(uHat .* phis)
    return uh
end

uh = pile_fem(p, 3)
fplot(uh, u, npoints=100)

In [ ]:
Nh = p.EA * uh'
fplot(Nh, N, npoints=100)

In [ ]:
fplot(0 * x, u - uh, npoints=100)

## Konvergenz

Fehlerfunktion

In [ ]:
fig = Figure()
Axis(fig[1, 1])
for k = 2 .^ (1:5)
    uh = pile_fem(p, k)
    fplot!(u - uh, npoints=100)
end
fig

Normalkraft

In [ ]:
fig = Figure()
Axis(fig[1, 1])
fplot!(N)
for k = 2 .^ (1:5)
    uh = pile_fem(p, k)
    Nh = p.EA * uh'
    fplot!(Nh, connect_jumps=true)
end
fig

Fehler in Energienorm

In [ ]:
nn = 5
hks = zeros(nn)
eks = zeros(nn)
nes = zeros(nn)

maxnorm(u) = maximum(abs.(u.(range(domain(u), 1000))))

for i in 1:nn
    k = 2^i
    uh = pile_fem(p, k)
    nes[i] = k
    hks[i] = p.l / k
    eks[i] = 100 * maxnorm(u - uh) / maxnorm(u)
end

In [ ]:
scatterlines(hks, eks)

In [ ]:
f = Figure()
Axis(f[1, 1], xscale=log10, yscale=log10, title="Fehler in Prozent")
scatterlines!(hks, eks)
f

In [ ]:
f = Figure()
Axis(f[1, 1], xscale=log10, yscale=log10, title="Fehler in Prozent")
scatterlines!(nes, eks)
f

# Aufgabe 4.3: Elastisch gelagerter Balken

In [ ]:
using Unitful
using CairoMakie
using LinearAlgebra

using MMJMesh.Plots
using MMJMesh.Mathematics

si(x, u) = ustrip(upreferred(x * uparse(u)))

## Beispiel Aufgabenblatt 1, Aufgabe 3

In [ ]:
fromunit(a) = ustrip(upreferred(a))

EI = si(210000, "N/mm^2") * si(8356, "cm^4")
l = si(8, "m")
q = si(50, "kN/m")
k = si(100, "kN/cm")

a(w, δw) = EI * integrate(w'' * δw'', 0 .. l) + k * (w(0) * δw(0) + w(l) * δw(l))
b(δw) = q * integrate(δw, 0 .. l)

phis = hermitefunctions(range(0, l, length=2), midnode=true)

K = [a(phii, phij) for phii = phis, phij = phis]
r = [b(phii) for phii = phis]

what = K \ r

w = what ⋅ phis
M = -EI * w'' / 1000
V = M'

In [ ]:
println(1000 * V(0) ≈ q * l / 2)
println(1000 * V(l) ≈ -q * l / 2)
println(abs(M(0)) < 1e-10)
println(abs(M(l)) < 1e-10)
println(1000 * M(l / 2) ≈ q * l^2 / 8)

In [ ]:
fig = Figure()
fplot!(Axis(fig[1, 1], title="Displacement"), 1000 * w, zero(w), npoints=10)
fplot!(Axis(fig[2, 1], title="Bending moment and shear force"), M, V, zero(w), npoints=10)
fig